# LoRA Fine-tuning on Colab

In [1]:
# Cell 1 — Install dependencies
!pip install transformers peft datasets accelerate bitsandbytes -q
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None — change runtime to GPU!')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB' if torch.cuda.is_available() else '')

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
# Cell 2 — Upload qa_pairs.json
from google.colab import files
import os

os.makedirs('data', exist_ok=True)
print('Upload your data/qa_pairs.json file:')
uploaded = files.upload()

for fname in uploaded:
    with open(f'data/{fname}', 'wb') as f:
        f.write(uploaded[fname])
    print(f'Saved: data/{fname}')

Upload your data/qa_pairs.json file:


Saving qa_pairs.json to qa_pairs (1).json
Saved: data/qa_pairs (1).json


In [3]:
# Cell 3 — Fine-tune Phi-3.5 Mini with LoRA (4-bit QLoRA)
import gc
import json
import os
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

# Clear any leftover GPU memory from previous runs
gc.collect()
torch.cuda.empty_cache()
print(f'Free VRAM: {round((torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9, 2)} GB')

BASE_MODEL = 'microsoft/Phi-3.5-mini-instruct'
ADAPTER_DIR = 'models/phi3-lora-adapter'
MAX_LENGTH = 256

SYSTEM_PROMPT = (
    'You are a helpful, caring, and professional customer support assistant for NUST Bank. '
    'Answer questions accurately based on NUST Bank products and services.'
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

LORA_CONFIG = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    bias='none',
)

def format_sample(pair):
    return (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\n{pair['question']}<|end|>\n"
        f"<|assistant|>\n{pair['answer']}<|end|>"
    )

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading model in 4-bit (QLoRA)...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={'': 0},
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LORA_CONFIG)
model.print_trainable_parameters()

print('Loading training data...')
with open('data/qa_pairs.json') as f:
    pairs = json.load(f)

formatted = [{'text': format_sample(p)} for p in pairs if p.get('question') and p.get('answer')]
print(f'Training samples: {len(formatted)}')
dataset = Dataset.from_list(formatted)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH, padding=False)

tokenized = dataset.map(tokenize, batched=True, remove_columns=['text'])
tokenized = tokenized.filter(lambda x: len(x['input_ids']) > 10)
print(f'Tokenized samples: {len(tokenized)}')

training_args = TrainingArguments(
    output_dir=ADAPTER_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=1,
    report_to='none',
    dataloader_drop_last=True,
    optim='paged_adamw_8bit',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print('Starting training...')
trainer.train()

os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Adapter saved to {ADAPTER_DIR}')

Free VRAM: 15.64 GB
Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Loading model in 4-bit (QLoRA)...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

trainable params: 1,572,864 || all params: 3,822,652,416 || trainable%: 0.0411
Loading training data...
Training samples: 496


Map:   0%|          | 0/496 [00:00<?, ? examples/s]

Filter:   0%|          | 0/496 [00:00<?, ? examples/s]

Tokenized samples: 496
Starting training...


Step,Training Loss
10,11.686546
20,9.435715
30,7.186388
40,6.084430
50,5.488774
60,5.163062
70,5.065374
80,4.763823
90,4.511295


Adapter saved to models/phi3-lora-adapter


In [4]:
# Cell 4 — Zip and download adapter
import shutil
from google.colab import files

shutil.make_archive('phi3-lora-adapter', 'zip', 'models/phi3-lora-adapter')
print('Downloading phi3-lora-adapter.zip (~60-80 MB)...')
files.download('phi3-lora-adapter.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>